In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"

con = duckdb.connect("olist.duckdb")

tables = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
}

for name, file in tables.items():
    path = (DATA_DIR / file).as_posix() 
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}')")

df = con.execute("SELECT COUNT(*) AS total_orders FROM orders").df()
print(df)


In [ ]:
df_revenue = con.execute("""
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp::TIMESTAMP) AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,   -- заказы, а не строки-позиции
        ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
    ORDER BY month
""").df()

print(df_revenue)


In [ ]:
tables = ['orders', 'order_items', 'customers', 'payments', 'reviews']

for table in tables: 
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

orders: 99,441 rows
order_items: 112,650 rows
customers: 99,441 rows
payments: 103,886 rows
reviews: 99,224 rows


In [ ]:
for table in tables:
    print(table)
    total = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    cols = con.execute(f"SELECT * FROM {table} LIMIT 0").df().columns

    for col in cols:
        nulls = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
        if nulls != 0:
            pct = round(nulls / total * 100, 1)
            print(f"  {col}: {nulls:,} nulls ({pct}%)")
